# Contextual Multi-Armed Bandit

For the contextual multi-armed bandit (cMAB) when user information is available (context), we implemented a generalisation of Thompson sampling algorithm ([Agrawal and Goyal, 2014](https://arxiv.org/pdf/1209.3352.pdf)) based on NumPyro.

![title](img/cmab.png)

The following notebook contains an example of usage of the class Cmab, which implements the algorithm above.

In [1]:
import numpy as np

from pybandits.cmab import CmabBernoulli
from pybandits.model import BayesianNeuralNetwork, BnnLayerParams, BnnParams, FeaturesConfig, StudentTArray

/home/runner/.cache/pypoetry/virtualenvs/pybandits-vYJB-miV-py3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
n_samples = 1000
n_features = 5

First, we need to define the input context matrix $X$ of size ($n\_samples, n\_features$) and the mapping of possible actions $a_i \in A$ to their associated model.

In [3]:
# context
X = 2 * np.random.random_sample((n_samples, n_features)) - 1  # random float in the interval (-1, 1)
print("X: context matrix of shape (n_samples, n_features)")
print(X[:10])

X: context matrix of shape (n_samples, n_features)
[[-0.02957493 -0.59143886  0.55228003 -0.64132776 -0.29741149]
 [ 0.67374229  0.33023622 -0.6267024   0.21046166 -0.38317251]
 [-0.41100593 -0.20097446 -0.48131722 -0.76488594 -0.34450623]
 [ 0.93619279 -0.28563061 -0.99368809 -0.94176735 -0.25133303]
 [ 0.66263248 -0.47096158 -0.11123706 -0.82827048 -0.27245218]
 [ 0.41328624 -0.54157872 -0.25812951 -0.82304854  0.93808103]
 [ 0.80754313 -0.29758982  0.80218089  0.26963617 -0.56447165]
 [-0.23822756  0.27564312  0.34758607  0.49514264  0.97572107]
 [-0.69815337 -0.01528131  0.98504657 -0.81029961  0.27197586]
 [ 0.51637145  0.56407867  0.69282722  0.18608986 -0.61921362]]


In [4]:
# define action model
bias = StudentTArray.cold_start(mu=1, sigma=2, shape=1)
weight = StudentTArray.cold_start(shape=(n_features, 1))
layer_params = BnnLayerParams(weight=weight, bias=bias)
model_params = BnnParams(bnn_layer_params=[layer_params])
feature_config = FeaturesConfig(n_features=n_features)

update_method = "VI"
update_kwargs = {"num_steps": 100, "batch_size": 128, "optimizer_type": "adam"}

actions = {
    "a1": BayesianNeuralNetwork(
        model_params=model_params,
        feature_config=feature_config,
        update_method=update_method,
        update_kwargs=update_kwargs,
    ),
    "a2": BayesianNeuralNetwork(
        model_params=model_params,
        feature_config=feature_config,
        update_method=update_method,
        update_kwargs=update_kwargs,
    ),
}

We can now init the bandit given the mapping of actions $a_i$ to their model.

In [5]:
# init contextual Multi-Armed Bandit model
cmab = CmabBernoulli(actions=actions)

The predict function below returns the action selected by the bandit at time $t$: $a_t = argmax_k P(r=1|\beta_k, x_t)$. The bandit selects one action per each sample of the contect matrix $X$.

In [6]:
# predict action
pred_actions, _, _ = cmab.predict(X)
print("Recommended action: {}".format(pred_actions[:10]))

Recommended action: ['a2', 'a1', 'a2', 'a2', 'a1', 'a2', 'a1', 'a2', 'a2', 'a1']


Now, we observe the rewards and the context from the environment. In this example rewards and the context are randomly simulated.

In [7]:
# simulate reward from environment
simulated_rewards = np.random.randint(2, size=n_samples).tolist()
print("Simulated rewards: {}".format(simulated_rewards[:10]))

Simulated rewards: [1, 0, 0, 1, 1, 1, 1, 1, 0, 0]


Finally, we update the model providing per each action sample: (i) its context $x_t$ (ii) the action $a_t$ selected by the bandit, (iii) the corresponding reward $r_t$.

In [8]:
# update model
cmab.update(context=X, actions=pred_actions, rewards=simulated_rewards)

SVI:   0%|          | 0/34 [00:00<?, ?it/s]

SVI:   3%|▎         | 1/34 [00:01<00:34,  1.03s/it]

SVI:   3%|▎         | 1/34 [00:01<00:34,  1.03s/it, loss=2330.6199]

SVI:   6%|▌         | 2/34 [00:01<00:33,  1.03s/it, loss=2340.7798]

SVI:   9%|▉         | 3/34 [00:01<00:32,  1.03s/it, loss=2570.4253]

...

SVI:  91%|█████████ | 31/34 [00:00<00:02,  1.23it/s, loss=2501.5764]

SVI:  94%|█████████▍| 32/34 [00:00<00:01,  1.23it/s, loss=2346.9641]

SVI:  97%|█████████▋| 33/34 [00:00<00:00,  1.23it/s, loss=3243.7927]

SVI: 100%|██████████| 34/34 [00:01<00:00, 24.21it/s, loss=3243.7927]

SVI: 100%|██████████| 34/34 [00:01<00:00, 24.21it/s, loss=2724.3779]